# **YouTube Tutorial AI Summarizer**

An AI-powered Python tool that extracts transcripts from YouTube tutorials and generates concise, structured summaries using LLMs (Gemini / GPT / Ollama).

> We use youtube_transcript_api which allows you to get the transcript text of any youtube video.












In [ ]:
!pip install youtube-transcript-api

In [2]:
#imports libraries
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter, SRTFormatter
import re
from openai import OpenAI
from dotenv import load_dotenv
#from google.colabg import userdata # dotenv equevilant for google colab
from IPython.display import Markdown, display, update_display
import requests
import ollama
import os

In [3]:
# models
model_gemini='gemini-3-flash-preview'
model_gpt = 'gpt-4o-mini'
model_llama = 'gemma4'


01069130719

In [4]:
#creating gemini client

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")


gemini=OpenAI(api_key=google_api_key,base_url=GEMINI_BASE_URL)
print("your gemini client is ready to use")

API key found and looks good so far!
your gemini client is ready to use


In [5]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [6]:
#creating Ollama client

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
print("your ollama client is ready to use")

your ollama client is ready to use


Initialize YouTube Transcript API and Formatters

In [7]:
ytt = YouTubeTranscriptApi()
formatter = TextFormatter() # --> Plain text
# formatter = SRTFormatter() # --> With timestamps



In [9]:
def get_video_id(url):
    """Extracts video ID from a YouTube URL."""
    regex = r"(?:v=|\/)([0-9A-Za-z_-]{11}).*"
    match = re.search(regex, url)
    if match:
        return match.group(1)
    raise ValueError("Invalid YouTube URL")


def get_transcript(url):
    video_id = get_video_id(url)
    fetched_transcript = ytt.fetch(video_id)
    # ^ defaults to English transcript, for other language use:
    # fetched = ytt.fetch(video_id, languages=['de', 'en'])
    transcript_text = formatter.format_transcript(fetched_transcript)
    return transcript_text

In [25]:
system_prompt="""Role: Senior Technical Instructor & Documentation Engineer

You are an expert Technical Instructor and Documentation Engineer specializing in transforming raw technical tutorials into clear, structured, and actionable learning materials.

Your task is to convert YouTube tutorial transcripts into high-quality Standard Operating Procedures (SOPs), Cheat Sheets, or Technical Guides that are easy to follow and reproduce.

## Core Objectives:

1. **Understand the Content**
   - Carefully analyze the full transcript.
   - Identify the main goal, workflow, and technical steps.

2. **Extract Prerequisites**
   - List all required tools, libraries, software, accounts, or setup steps mentioned.

3. **Convert into Step-by-Step Guide**
   - Rewrite the tutorial as a clear numbered workflow.
   - Ensure each step is actionable, precise, and unambiguous.
   - Preserve the original logical order of the tutorial.

4. **Preserve Technical Accuracy**
   - Do NOT modify code, commands, parameters, or configurations.
   - Keep all technical details exactly as stated.

5. **Identify Key Insights**
   - Highlight important explanations, concepts, or reasoning behind steps.

6. **Highlight Warnings and Gotchas**
   - Clearly mention mistakes, pitfalls, or troubleshooting tips provided in the tutorial.

7. **Best Practices**
   - Include optimization tips or recommendations if mentioned or implied.

## Output Format (Strict Markdown):

- Use **Numbered Lists** for main steps.
- Use **Code Blocks** for:
  - Commands
  - Code snippets
  - Configuration files
- Use **Bold text** for UI elements or important labels (e.g., "Click the Run button").
- Use **Blockquotes (> )** for tips, warnings, or pro insights.
- Keep output clean, structured, and easy to scan.

## Writing Style:
- Professional, concise, and instructional
- No unnecessary storytelling
- Focus on clarity and reproducibility
"""

In [28]:
def get_user_prompt(url):
    prompt = f"""You are given a full transcript of a YouTube technical tutorial.

Your task is to carefully analyze it and convert it into a structured, easy-to-follow technical guide (SOP / Cheat Sheet).

## Instructions:
- Extract and organize the key steps in a logical sequence.
- Rewrite explanations into clear actionable instructions.
- Preserve all technical details exactly (code, commands, configurations).
- Remove redundancy and filler speech.
- Ensure the final output is well-formatted in Markdown.

## Output Requirements:
1. Start with a short overview of what the tutorial is about.
2. List prerequisites (if any).
3. Provide a step-by-step guide.
4. Include important notes, warnings, and tips.
5. End with a concise summary of the workflow.

## Transcript:
{get_transcript(url)}"""
   
    return prompt

In [29]:

def summerize_video(url):
    user_prompt = get_user_prompt(url)
    stream = ollama.chat.completions.create(
        model=model_llama,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        stream = True,
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [30]:
summerize_video("https://www.youtube.com/watch?v=VyWAvY2CF9c")

Here is a structured summary of the provided text, covering the key concepts taught.

## Overview of Deep Learning Workflow

The presentation provides a comprehensive walkthrough of the typical machine learning and deep learning pipeline, from initial setup to final model evaluation and refinement.

---

### 1. Data Preparation and Preprocessing

While not exhaustively detailed, the context implies the need for clean, structured data, which is fundamental to any ML process.

---

### 2. Model Training and Iteration (The Core Loop)

The goal is to train a model that can generalize from training data to unseen data.

*   **Model Architecture:** Decisions must be made regarding the structure of the neural network.
*   **Optimization:** The model learns by adjusting weights and biases to minimize a **Loss Function**.
*   **Epochs:** The model cycles through the entire dataset multiple times (epochs) to refine its parameters.

---

### 3. Model Evaluation and Performance Metrics

It is crucial to assess how well the model performs on unseen data.

*   **Train vs. Test Sets:** Data is split to prevent **Overfitting**.
    *   **Training Set:** Used to teach the model.
    *   **Test Set:** Used to evaluate the final model performance.
*   **Overfitting:** Occurs when the model learns the noise and specifics of the training data *too* well, performing poorly on new data.
*   **Underfitting:** Occurs when the model is too simple to capture the underlying patterns in the data.

---

### 4. Advanced Techniques for Robustness and Improvement

The material dives into specific techniques used to improve model performance and prevent common pitfalls.

#### A. Regularization Techniques (Combating Overfitting)
These methods penalize complexity during training:
*   **Dropout:** During training, random units are temporarily "dropped out" (ignored) in the network. This forces the network to learn redundant representations, making it less reliant on any single feature or unit.
*   **L1/L2 Regularization:** Adding penalty terms to the loss function to discourage overly large weights.

#### B. Data Augmentation
*   **Concept:** Creating new, synthetic training samples by applying slight, realistic transformations to existing data (e.g., rotating, cropping, or changing brightness of images). This increases the effective size and variability of the training set.

---

### 5. Key Topics in Deep Dive (Specific Considerations)

The presentation touches upon several critical, specialized aspects of data handling and model refinement:

*   **Normalization/Standardization:** Scaling features to a standard range ($\mu=0, \sigma=1$) to help the optimization algorithm (like Gradient Descent) converge faster and more reliably.
*   **Hyperparameter Tuning:** The process of optimizing settings that are *not* learned by the model (e.g., learning rate, number of layers, batch size). Grid search and random search are common tuning methods.
*   **Transfer Learning:** Utilizing a model that was pre-trained on a massive, general dataset (like ImageNet) and then fine-tuning it for a specific, smaller task. This saves significant computational time and data requirements.
*   **Bias-Variance Tradeoff:** A fundamental concept:
    *   **High Bias (Underfitting):** Model is too simple; consistent errors.
    *   **High Variance (Overfitting):** Model is too complex; highly sensitive to small fluctuations in the training data. The goal is to find the balance point.
*   **Curse of Dimensionality:** The problem where the volume of the data space increases so rapidly with the number of dimensions that the available data becomes sparse, making accurate modeling difficult.